# Лекција 11 - Протокол агент-агент (A2A)


## Подешавање


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Шта је A2A протокол?

The **Agent-to-Agent (A2A) протокол** је отворени стандард који омогућава AI агентима да комуницирају,
откривају једни друге, и сарађују — чак и када су изграђени на различитим оквирима или хостовани
од стране различитих сервиса.

Кључни појмови:

- **Откривање** – Агенти објављују *картицу агента* која описује њихове могућности, олакшавајући
  другим агентима (или оркестраторима) да пронађу правог специјалисту за задатак.
- **Размена порука** – Агенти размењују структурисане поруке преко заједничког протокола, тако да
  захтев од једног агента може бити разумљив и испуњен од стране другог без обзира на његову унутрашњу
  имплементацију.
- **Животни циклус задатка** – A2A дефинише стања као што су *поднето*, *у току*, *завршено* и
  *неуспело*, дајући оркестратору пуну видљивост у томе како делегирани задатак напредује.

У овој лекцији симулирамо сарадњу у A2A стилу повезивањем три специјализована агента за путовања
у радни ток где сваки агент доприноси својом стручношћу и прослеђује резултате следећем агенту.


## Креирање специјализованих туристичких агената


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Сарадња више агената кроз ток посла

Повезујемо три агента у секвенцијални ток посла који огледа A2A размену порука:

1. **CurrencyExchangeAgent** прима кориснички захтев и пружа смернице о валути.
2. **ActivityPlannerAgent** прима обогаћен контекст и додаје препоруке за активности.
3. **TravelManagerAgent** синтетише оба уноса у коначни бриф о путовању.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Разумевање A2A у продукцији

У продукционом окружењу A2A протокол омогућава моћне сценарије међу сервисима:

| Могућност | Опис |
|---|---|
| **Интероперабилност међу фрејмворцима** | Агент направљен у једном фрејмворку може делегирати задатке агенту направљеном у било ком другом A2A-усклађеном фрејмворку, омогућавајући праву интероперабилност између организација. |
| **Границе сервиса** | Агенти могу да постоје у одвојеним микросервисима, облачним регионима или чак у различитим организацијама, а да ипак беспрекорно сарађују. |
| **Динамичко откривање** | Оркестратор може током извршавања упитати регистар Agent Card-ова да пронађе најприкладнијег специјалисту за дат подзадатак. |
| **Стриминг & push обавештења** | A2A подржава Server-Sent Events (SSE) за ажурирања напретка у реалном времену и push обавештења за дуготрајне задатке. |

Радни ток који смо изградили горе је поједностављена, у-процесу верзија овог шаблона. У реалном распоређивању сваки агент би излагао HTTP крајњу тачку, објавио Agent Card, и комуницирао путем A2A JSON-RPC протокола.


## Резиме

У овој лекцији сте научили:

1. **Шта је A2A протокол** — отворени стандард за откривање агената, размену порука,
   и управљање задацима.
2. **Како креирати специјализоване агенте** — агента за размену валута, агента за планирање активности,
   и оркестратор за управљање путовањима.
3. **Како повезати агенте у ток рада** — користећи `WorkflowBuilder` да бисте моделовали секвенцијално
   прослеђивање порука између агената.
4. **Како A2A функционише у продукцији** — омогућавајући сарадњу између различитих фрејмворка и сервиса
   уз динамично откривање и стримовање ажурирања.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за превод која користи вештачку интелигенцију [Co-op Translator](https://github.com/Azure/co-op-translator). Иако настојимо да обезбедимо тачност, имајте у виду да аутоматизовани преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитетним извором. За кључне информације препоручује се професионални људски превод. Не сносимо одговорност за било каква непоразумевања или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
